In [37]:
import pandas as pd

df = pd.read_csv('borg_traces_data.csv')

df.dropna(inplace=True)

# print(df.iloc[0])
# print(', '.join(map(str, df.iloc[0].values)))

print(df[['priority', 'resource_request', 'average_usage', 'maximum_usage']].head())

   priority                                   resource_request  \
2       103  {'cpus': 0.048583984375, 'memory': 0.004165649...   
3       200  {'cpus': 0.0704345703125, 'memory': 0.04162597...   
6       117  {'cpus': 0.00566864013671875, 'memory': 0.0015...   
7       103  {'cpus': 0.022003173828125, 'memory': 0.001501...   
8         0  {'cpus': 0.0081024169921875, 'memory': 0.00361...   

                                       average_usage  \
2  {'cpus': 0.024200439453125, 'memory': 0.002788...   
3  {'cpus': 0.047607421875, 'memory': 0.034423828...   
6  {'cpus': 0.0042572021484375, 'memory': 0.00131...   
7  {'cpus': 0.0064697265625, 'memory': 0.00075817...   
8  {'cpus': 0.0002002716064453125, 'memory': 0.00...   

                                       maximum_usage  
2  {'cpus': 0.06005859375, 'memory': 0.0028457641...  
3   {'cpus': 0.13330078125, 'memory': 0.03466796875}  
6  {'cpus': 0.012481689453125, 'memory': 0.001653...  
7  {'cpus': 0.027984619140625, 'memory': 0.000

In [38]:
print(df["scheduling_class"].value_counts())

scheduling_class
2    99956
1    81550
0    72154
3    27217
Name: count, dtype: int64


In [39]:
print(df.shape)

(280877, 34)


In [40]:
newDf = df.copy()
newDf.drop(columns=["no","time","instance_events_type","collection_id","collection_type","alloc_collection_id","instance_index","machine_id","collections_events_type","user","collection_name","collection_logical_name",
                 "start_after_collection_ids","vertical_scaling","scheduler", "cluster", "event", "failed", "constraint"], inplace=True)

print(newDf.columns)

Index(['scheduling_class', 'priority', 'resource_request', 'start_time',
       'end_time', 'average_usage', 'maximum_usage', 'random_sample_usage',
       'assigned_memory', 'page_cache_memory', 'cycles_per_instruction',
       'memory_accesses_per_instruction', 'sample_rate',
       'cpu_usage_distribution', 'tail_cpu_usage_distribution'],
      dtype='object')


In [41]:
# execution time calculation. (end_time - start_time)
import pandas as pd

def add_execution_time_ms(df):
    """
    Adds 'execution_time' column (in milliseconds) to the DataFrame.
    Assumes 'start_time' and 'end_time' are in nanoseconds.

    Modifies the DataFrame in-place by adding the 'execution_time' column.
    """
    if 'start_time' not in df.columns or 'end_time' not in df.columns:
        raise ValueError("DataFrame must contain 'start_time' and 'end_time' columns.")

    df['execution_time'] = (df['end_time'] - df['start_time']) / 1e6  # milliseconds
    return df

newDf = add_execution_time_ms(newDf)

print(newDf.head())

print(newDf["execution_time"].unique())

   scheduling_class  priority  \
2                 2       103   
3                 3       200   
6                 1       117   
7                 2       103   
8                 0         0   

                                    resource_request     start_time  \
2  {'cpus': 0.048583984375, 'memory': 0.004165649...    81300000000   
3  {'cpus': 0.0704345703125, 'memory': 0.04162597...  1075500000000   
6  {'cpus': 0.00566864013671875, 'memory': 0.0015...   343800000000   
7  {'cpus': 0.022003173828125, 'memory': 0.001501...   455400000000   
8  {'cpus': 0.0081024169921875, 'memory': 0.00361...  2249100000000   

        end_time                                      average_usage  \
2    81600000000  {'cpus': 0.024200439453125, 'memory': 0.002788...   
3  1075800000000  {'cpus': 0.047607421875, 'memory': 0.034423828...   
6   344100000000  {'cpus': 0.0042572021484375, 'memory': 0.00131...   
7   455700000000  {'cpus': 0.0064697265625, 'memory': 0.00075817...   
8  2249400000000  {

In [42]:
import numpy as np
import pandas as pd
import ast  # To safely parse stringified dictionaries

def safe_log_transform(x, epsilon=1e-9):
    return np.log(x + epsilon)

def normalize_column(col):
    min_val = np.min(col)
    max_val = np.max(col)
    if max_val == min_val:
        return np.zeros_like(col)
    return (col - min_val) / (max_val - min_val)

def extract_feature(df, col, key):
    return df[col].apply(lambda x: ast.literal_eval(x)[key] if pd.notna(x) else 0.0)

def normalize_dataset(df, input_cols, output_cols):
    norm_data = pd.DataFrame()

    # Handle special parsing for resource_request and average_usage
    if 'resource_request' in input_cols:
        df['resource_request_cpus'] = extract_feature(df, 'resource_request', 'cpus')
        df['resource_request_memory'] = extract_feature(df, 'resource_request', 'memory')
        input_cols.remove('resource_request')
        input_cols += ['resource_request_cpus', 'resource_request_memory']
    
    if 'average_usage' in input_cols:
        df['average_usage_cpus'] = extract_feature(df, 'average_usage', 'cpus')
        df['average_usage_memory'] = extract_feature(df, 'average_usage', 'memory')
        input_cols.remove('average_usage')
        input_cols += ['average_usage_cpus', 'average_usage_memory']

    # Normalize all selected columns
    for col in input_cols + output_cols:
        if col not in df.columns:
            raise ValueError(f"Missing column in dataset: {col}")
        log_transformed = safe_log_transform(df[col].astype(float).values)
        normalized = normalize_column(log_transformed)
        norm_data[col] = normalized

    return norm_data


In [ ]:
input_features = ["resource_request", "average_usage"]
output_features = ["priority", "execution_time", "cycles_per_instruction"]

normalized_df = normalize_dataset(newDf, input_features, output_features)
print(normalized_df.head())


   resource_request_cpus  resource_request_memory  average_usage_cpus  \
0               0.886052                 0.782717            0.885305   
1               0.904645                 0.900921            0.920537   
2               0.778500                 0.732351            0.794818   
3               0.846397                 0.730304            0.816611   
4               0.796383                 0.775402            0.635653   

   average_usage_memory  priority  execution_time  cycles_per_instruction  
0              0.783431  0.945047             1.0                0.217690  
1              0.916100  0.969778             1.0                0.278833  
2              0.743794  0.949797             1.0                0.155695  
3              0.714681  0.945047             1.0                0.322441  
4              0.708712  0.000000             1.0                0.435326  


In [44]:
print(normalized_df.shape)
print(normalized_df['execution_time'].unique())

(280877, 7)
[1.         0.19261118 0.12152413 0.34116135 0.99705334 0.97287653
 0.         0.76606145 0.96305591 0.74979474 0.65942208 0.87612252
 0.24304825 0.96520713 0.38522237 0.40369441 0.51622568 0.97625471
 0.28217028 0.99764665 0.54972191 0.90946417 0.9189949  0.85338711
 0.60205438 0.78094621 0.85203326 0.99465982 0.87730315 0.8318922
 0.93409526 0.76156576 0.85870031 0.31413531 0.4496927  0.90650091
 0.7747211  0.4860965  0.96873486 0.82249769 0.9940563  0.43565943
 0.95718684 0.96943197 0.9044972  0.89088327 0.97219301 0.91806481
 0.91237818 0.60762063 0.97081793 0.81426508 0.90247033 0.61824948
 0.86637989 0.96733228 0.71782972 0.9448242  0.92803413 0.95493427
 0.73717619 0.89729821 0.46268548 0.87253218 0.92978737 0.98601543
 0.92537096 0.8208818  0.54192853 0.47478147 0.98538135 0.86000383
 0.89196886 0.82409881 0.71188602 0.85606384 0.84510263 0.90749424
 0.89518587 0.7235785  0.75221302 0.81594282 0.87847587 0.80384683
 0.49672535 0.62333164 0.94878198 0.92357277 0.9847

In [29]:
# model for priority, execution_time, cpi prediction
import torch
import torch.nn as nn

class MultiModalRegressor(nn.Module):
    def __init__(self):
        super(MultiModalRegressor, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),       # Helps with small value training

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 3)  # Output: priority, execution_time, cpi
        )

    def forward(self, x):
        return self.model(x)


In [49]:
sample = MultiModalRegressor()

total_params = sum(p.numel() for p in sample.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")


Total trainable parameters: 19795


In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Assuming normalized_df is defined already
input_cols = ['resource_request_cpus', 'resource_request_memory', 'average_usage_cpus', 'average_usage_memory']
output_cols = ['priority', 'execution_time', 'cycles_per_instruction']

X = torch.tensor(normalized_df[input_cols].values, dtype=torch.float32)
y = torch.tensor(normalized_df[output_cols].values, dtype=torch.float32)

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = MultiModalRegressor()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    preds = []
    targets = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)

            preds.append(outputs.numpy())
            targets.append(batch_y.numpy())

    val_loss /= len(val_loader.dataset)

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    mae = mean_absolute_error(targets, preds)
    r2 = r2_score(targets, preds)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | MAE: {mae:.6f} | R²: {r2:.4f}")


Epoch 01 | Train Loss: 0.035146 | Val Loss: 0.015897 | MAE: 0.067982 | R²: 0.7244
Epoch 02 | Train Loss: 0.016577 | Val Loss: 0.014419 | MAE: 0.054802 | R²: 0.7480
Epoch 03 | Train Loss: 0.015246 | Val Loss: 0.015323 | MAE: 0.057948 | R²: 0.7464
Epoch 04 | Train Loss: 0.014454 | Val Loss: 0.015226 | MAE: 0.053093 | R²: 0.7513
Epoch 05 | Train Loss: 0.013785 | Val Loss: 0.012749 | MAE: 0.054010 | R²: 0.7567
Epoch 06 | Train Loss: 0.013170 | Val Loss: 0.011517 | MAE: 0.047903 | R²: 0.7964
Epoch 07 | Train Loss: 0.012623 | Val Loss: 0.009434 | MAE: 0.039752 | R²: 0.8130
Epoch 08 | Train Loss: 0.012428 | Val Loss: 0.011575 | MAE: 0.044873 | R²: 0.7881
Epoch 09 | Train Loss: 0.011813 | Val Loss: 0.008967 | MAE: 0.038155 | R²: 0.8277
Epoch 10 | Train Loss: 0.011453 | Val Loss: 0.009959 | MAE: 0.039390 | R²: 0.8193
Epoch 11 | Train Loss: 0.011205 | Val Loss: 0.009418 | MAE: 0.037790 | R²: 0.8230
Epoch 12 | Train Loss: 0.010911 | Val Loss: 0.009047 | MAE: 0.037844 | R²: 0.8211
Epoch 13 | Train

Has a average r2 score of 0.82 to 0.84 for time_quantum_pred_scheduler_model.pth

In [53]:
import torch

torch.save(model.state_dict(), "time_quantum_pred_scheduler_model.pth")
print("Model weights saved to 'time_quantum_pred_scheduler_model.pth'")


Model weights saved to 'time_quantum_pred_scheduler_model.pth'


In [10]:
# model for resource request predition
import torch
import torch.nn as nn

class MultiModalRegressorNewModel(nn.Module):
    def __init__(self):
        super(MultiModalRegressorNewModel, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),  # Fixed from 64 to 128

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 2)
        )

    def forward(self, x):
        return self.model(x)


In [70]:
sample2 = MultiModalRegressorNewModel()
total_params2 = sum(p.numel() for p in sample2.parameters() if p.requires_grad)

print(f"Total trainable parameters: {total_params2}")

Total trainable parameters: 11658


In [71]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Assuming normalized_df is defined already
input_cols = ['average_usage_cpus', 'average_usage_memory']
output_cols = ['execution_time', 'cycles_per_instruction']

X = torch.tensor(normalized_df[input_cols].values, dtype=torch.float32)
y = torch.tensor(normalized_df[output_cols].values, dtype=torch.float32)

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Model, loss, optimizer
model2 = MultiModalRegressorNewModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)

# Training Loop
epochs = 50
for epoch in range(epochs):
    model2.train()
    train_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model2(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model2.eval()
    val_loss = 0.0
    preds = []
    targets = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model2(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)

            preds.append(outputs.cpu().numpy())   # fixed
            targets.append(batch_y.cpu().numpy()) # fixed

    val_loss /= len(val_loader.dataset)

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    mae = mean_absolute_error(targets, preds)
    r2 = r2_score(targets, preds)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | MAE: {mae:.6f} | R²: {r2:.4f}")

Epoch 01 | Train Loss: 0.020168 | Val Loss: 0.012898 | MAE: 0.063273 | R²: 0.6315
Epoch 02 | Train Loss: 0.009953 | Val Loss: 0.008423 | MAE: 0.055079 | R²: 0.6899
Epoch 03 | Train Loss: 0.008933 | Val Loss: 0.008531 | MAE: 0.048455 | R²: 0.6999
Epoch 04 | Train Loss: 0.008531 | Val Loss: 0.007505 | MAE: 0.048806 | R²: 0.6991
Epoch 05 | Train Loss: 0.008155 | Val Loss: 0.007307 | MAE: 0.047595 | R²: 0.7236
Epoch 06 | Train Loss: 0.007954 | Val Loss: 0.008165 | MAE: 0.052670 | R²: 0.7078
Epoch 07 | Train Loss: 0.007794 | Val Loss: 0.008535 | MAE: 0.044528 | R²: 0.7018
Epoch 08 | Train Loss: 0.007588 | Val Loss: 0.008376 | MAE: 0.045189 | R²: 0.7168
Epoch 09 | Train Loss: 0.007389 | Val Loss: 0.008836 | MAE: 0.044349 | R²: 0.7160
Epoch 10 | Train Loss: 0.007268 | Val Loss: 0.006854 | MAE: 0.040128 | R²: 0.7498
Epoch 11 | Train Loss: 0.007339 | Val Loss: 0.006363 | MAE: 0.039021 | R²: 0.7502
Epoch 12 | Train Loss: 0.007101 | Val Loss: 0.005964 | MAE: 0.040963 | R²: 0.7499
Epoch 13 | Train

Has a r2 score of 0.78 to 0.8 for time_quantum_pred_scheduler_model2.pth

In [72]:
import torch

torch.save(model2.state_dict(), "time_quantum_pred_scheduler_model2.pth")
print("Model weights saved to 'time_quantum_pred_scheduler_model2.pth'")

Model weights saved to 'time_quantum_pred_scheduler_model2.pth'


Formula 1: timeQuantum = priority * (execution_time/ cpi) <br>
Formula 2: timeQuantum = scalalingFactor * priortiy * (execution_time/ cpi) -> scalaling factor yet to be determined 